# RL + LLM Training (Colab) for Adaptive Traffic Intelligence

This notebook trains:
1. **RL traffic controller** (PPO) on the `TrafficEnv`
2. **LLM policy model** (LoRA fine-tuning) to imitate the RL controller

It also compares **Fixed baseline vs RL vs LLM** on waiting time, queue length, and throughput.

## Best Model Choices (Practical)

### RL (this task)
- **Recommended default:** `PPO` (stable, strong for this discrete-control setup)
- Alternatives: `DQN` (already in repo), `A2C`

### LLM policy model
- **Best on Colab T4 (recommended):** `Qwen/Qwen2.5-1.5B-Instruct` with LoRA + 4-bit
- Higher quality (needs stronger GPU): `meta-llama/Meta-Llama-3.1-8B-Instruct`
- Faster/smaller fallback: `Qwen/Qwen2.5-0.5B-Instruct`

In [ ]:
# ==== 0) Runtime setup ====
!nvidia-smi

!pip -q install -U gymnasium stable-baselines3 sb3-contrib
!pip -q install -U transformers datasets peft trl accelerate bitsandbytes sentencepiece


In [ ]:
# ==== 1) Get project code ====
import os
from pathlib import Path

REPO_URL = "https://github.com/DivyankLosse/TRLE-Hackethon.git"
PROJECT_DIR = Path("TRLE-Hackethon")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL}

%cd TRLE-Hackethon

# Optional: install local package deps
!pip -q install -r requirements.txt


In [ ]:
# ==== 2) Imports and Gym wrapper for TrafficEnv ====
import math
import json
import random
from dataclasses import dataclass

import numpy as np
import gymnasium as gym
from gymnasium import spaces

from traffic_rl.env.traffic_env import TrafficEnv
from traffic_rl.baseline.fixed_time_controller import FixedTimeController

class TrafficGymEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, env_config: dict):
        super().__init__()
        self.base_config = dict(env_config)
        self.base_env = TrafficEnv(config=self.base_config)
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=np.zeros((10,), dtype=np.float32),
            high=np.full((10,), np.finfo(np.float32).max, dtype=np.float32),
            dtype=np.float32,
        )

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            cfg = dict(self.base_config)
            cfg["seed"] = int(seed)
            self.base_env = TrafficEnv(config=cfg)
        obs = self.base_env.reset().astype(np.float32)
        return obs, {}

    def step(self, action):
        obs, reward, done, info = self.base_env.step(int(action))
        obs = obs.astype(np.float32)
        terminated = bool(done)
        truncated = False
        return obs, float(reward), terminated, truncated, info


In [ ]:
# ==== 3) Train RL model (PPO) ====
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

ENV_CONFIG = {
    "max_steps": 120,
    "arrival_mode": "stochastic",
    "lane_bias": (1.6, 0.8, 1.4, 0.6),
    "peak_rates": (3.8, 2.2, 3.4, 1.5),
    "offpeak_rates": (1.4, 0.9, 1.2, 0.7),
    "peak_duration": 35,
    "cycle_duration": 60,
    "service_rate": 2,
    "ambulance_spawn_prob": 0.08,
    "seed": 42,
}

N_ENVS = 4
TOTAL_TIMESTEPS = 120_000   # increase to 300k+ for stronger policy

def make_env(rank):
    def _thunk():
        cfg = dict(ENV_CONFIG)
        cfg["seed"] = ENV_CONFIG["seed"] + rank
        return TrafficGymEnv(cfg)
    return _thunk

vec_env = DummyVecEnv([make_env(i) for i in range(N_ENVS)])

ppo = PPO(
    policy="MlpPolicy",
    env=vec_env,
    n_steps=1024,
    batch_size=256,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    verbose=1,
    seed=42,
)

ppo.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)

Path("artifacts").mkdir(exist_ok=True)
ppo.save("artifacts/ppo_traffic")
print("Saved RL model -> artifacts/ppo_traffic.zip")

In [ ]:
# ==== 4) Evaluate baseline vs RL ====
def eval_policy(policy_fn, env_config, episodes=20):
    metrics = {
        "reward": [],
        "avg_queue_length": [],
        "avg_waiting_time": [],
        "throughput": [],
        "ambulance_clearances": [],
    }

    for ep in range(episodes):
        cfg = dict(env_config)
        cfg["seed"] = env_config.get("seed", 42) + ep
        env = TrafficEnv(config=cfg)
        state = env.reset()
        done = False

        rewards, queues, waits, throughputs = [], [], [], []
        amb_clears = 0
        step = 0

        while not done:
            action = int(policy_fn(state, step))
            state, reward, done, info = env.step(action)
            rewards.append(float(reward))
            queues.append(float(info["queue_sum"]))
            waits.append(float(info["waiting_sum"]))
            throughputs.append(float(info["throughput"]))
            amb_clears += int(bool(info.get("ambulance_cleared", False)))
            step += 1

        metrics["reward"].append(sum(rewards))
        metrics["avg_queue_length"].append(float(np.mean(queues) if queues else 0.0))
        metrics["avg_waiting_time"].append(float(np.mean(waits) if waits else 0.0))
        metrics["throughput"].append(sum(throughputs))
        metrics["ambulance_clearances"].append(float(amb_clears))

    return {k: float(np.mean(v)) for k, v in metrics.items()}

fixed = FixedTimeController(switch_interval=5)
baseline_metrics = eval_policy(lambda _s, t: fixed.action_for_step(t), ENV_CONFIG, episodes=20)
rl_metrics = eval_policy(lambda s, _t: ppo.predict(s, deterministic=True)[0], ENV_CONFIG, episodes=20)

def pct_improve(lower_better_key):
    b, r = baseline_metrics[lower_better_key], rl_metrics[lower_better_key]
    return 0.0 if b == 0 else ((b - r) / b) * 100.0

def pct_gain(higher_better_key):
    b, r = baseline_metrics[higher_better_key], rl_metrics[higher_better_key]
    return 0.0 if b == 0 else ((r - b) / b) * 100.0

improvement = {
    "waiting_time_improvement_pct": pct_improve("avg_waiting_time"),
    "queue_length_improvement_pct": pct_improve("avg_queue_length"),
    "throughput_gain_pct": pct_gain("throughput"),
    "ambulance_clearance_gain_pct": pct_gain("ambulance_clearances"),
}

print("Baseline:", json.dumps(baseline_metrics, indent=2))
print("RL:", json.dumps(rl_metrics, indent=2))
print("Improvement:", json.dumps(improvement, indent=2))

In [ ]:
# ==== 5) Build policy dataset from RL trajectories ====
from datasets import Dataset

def state_to_prompt(state):
    q = [int(x) for x in state[:4]]
    w = [int(x) for x in state[4:8]]
    phase = int(state[8])
    ambulance = int(state[9])
    return (
        "You control a traffic signal. Output only one action token: 0, 1, or 2.\n"
        f"Queues(Q1..Q4)={q}\n"
        f"Waiting(W1..W4)={w}\n"
        f"CurrentPhase={phase} AmbulanceFlag={ambulance}\n"
        "Action:"
    )

def collect_policy_examples(model, env_config, episodes=30):
    rows = []
    for ep in range(episodes):
        cfg = dict(env_config)
        cfg["seed"] = env_config.get("seed", 42) + 1000 + ep
        env = TrafficEnv(config=cfg)
        s = env.reset()
        done = False

        while not done:
            a, _ = model.predict(s, deterministic=True)
            a = int(a)
            prompt = state_to_prompt(s)
            text = f"### Instruction:\n{prompt}\n### Response:\n{a}"
            rows.append({"prompt": prompt, "action": str(a), "text": text})
            s, _r, done, _info = env.step(a)

    return rows

policy_rows = collect_policy_examples(ppo, ENV_CONFIG, episodes=35)
print(f"Collected {len(policy_rows)} examples")

dataset = Dataset.from_list(policy_rows).train_test_split(test_size=0.05, seed=42)
dataset

## LLM Fine-Tuning (LoRA)
Default model is selected for Colab T4 stability. If OOM happens, switch to `Qwen/Qwen2.5-0.5B-Instruct`.

In [ ]:
# ==== 6) Load base LLM + LoRA setup ====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

BASE_LLM = "Qwen/Qwen2.5-1.5B-Instruct"  # recommended on Colab T4
# BASE_LLM = "Qwen/Qwen2.5-0.5B-Instruct"  # fallback if VRAM is tight

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_LLM, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    BASE_LLM,
    device_map="auto",
    quantization_config=bnb_config,
)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

llm = get_peft_model(llm, lora_cfg)
llm.print_trainable_parameters()

In [ ]:
# ==== 7) Fine-tune LLM on RL policy traces ====
from trl import SFTTrainer, SFTConfig

sft_cfg = SFTConfig(
    output_dir="artifacts/llm_lora_policy",
    max_seq_length=512,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    save_strategy="epoch",
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=llm,
    args=sft_cfg,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    formatting_func=lambda ex: ex["text"],
)

trainer.train()
trainer.model.save_pretrained("artifacts/llm_lora_policy")
tokenizer.save_pretrained("artifacts/llm_lora_policy")
print("Saved LLM adapter -> artifacts/llm_lora_policy")

In [ ]:
# ==== 8) Evaluate LLM policy ====
import re

@torch.inference_mode()
def llm_action_from_state(state):
    prompt = state_to_prompt(state)
    wrapped = f"### Instruction:\n{prompt}\n### Response:\n"
    inputs = tokenizer(wrapped, return_tensors="pt").to(llm.device)
    out = llm.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"[012]", gen)
    return int(m.group(0)) if m else 0

llm_metrics = eval_policy(lambda s, _t: llm_action_from_state(s), ENV_CONFIG, episodes=10)
print("LLM metrics:")
print(json.dumps(llm_metrics, indent=2))

In [ ]:
# ==== 9) Save summary ====
summary = {
    "baseline": baseline_metrics,
    "rl": rl_metrics,
    "llm": llm_metrics,
    "rl_vs_baseline": improvement,
    "base_llm": BASE_LLM,
}

Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/colab_training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved -> artifacts/colab_training_summary.json")

## Optional: push checkpoints to Hugging Face Hub

After training, you can push:
- RL checkpoint: `artifacts/ppo_traffic.zip`
- LLM adapter: `artifacts/llm_lora_policy/`

Use `huggingface_hub` or git-lfs depending on your target repo layout.